# Choosing settings: grid size, proposal mode, and batch runs

Given your grid size, dimension, and compute
budget, this notebook recommends what settings should you pick. It runs small, fast configurations live so
every number below comes from actually running the simulator, and cites the larger
confluent-density benchmarks from `crates/cpm-core/benches/` since running those
live would take too long for a notebook.

Requires the package built and installed (from the repo root):

```bash
maturin develop --release
```

In [ ]:
import time

import cpm

# A small 2D system, the same shape as the README quickstart.
sim = cpm.CPM(
    grid=(30, 30),
    boundary="periodic",
    seed=1,
    copy_neighborhood="von_neumann",
    connectivity_neighborhood="von_neumann",
)
sim.register_cell_type(
    name="epithelial",
    target_volume=50,
    target_interface=75,
    lambda_volume=10.0,
    lambda_interface=2.0,
)
sim.add_cells(cell_type="epithelial", n=5)
sim.set_adhesion([[0.0, 5.0], [5.0, 2.0]])
sim.initialize(warn_on_default_init=False)

result = sim.run(burn_in_mcs=500, readout_mcs=500, sampling_interval_mcs=100)
print(f"density: {result.metadata.derived_phi:.3f}")
print(f"status: {result.status.kind}")

density: 0.278
status: Ok


## Performance Factors

One MCS consists of `N_sites` attempts, where `N_sites` represents the number of interior lattice sites. Both `burn_in_mcs` and `readout_mcs` execute that many attempts per cycle. The `sampling_interval_mcs` parameter sets the recording frequency during the readout window without changing the number of attempts executed.

Execution speed depends on total site count based on grid dimensions alongside total MCS volume.

Initializing the 3D connectivity lookup table requires a one-time setup step generating a shared 8 MiB allocation. Subsequent 3D simulations within the same process reuse this table without repeating the initialization overhead. A preliminary run pre-allocates this table so following benchmarks reflect true simulation time.

In [2]:
def mcs_per_second(grid, proposal="uniform", n_mcs=2000, seed=1, n_cells=10):
    dim = len(grid)
    sim = cpm.CPM(
        grid=grid,
        boundary="periodic",
        seed=seed,
        copy_neighborhood="von_neumann",
        connectivity_neighborhood="von_neumann",
        proposal=proposal,
    )
    sim.register_cell_type(
        name="a",
        target_volume=30 if dim == 2 else 27,
        target_interface=40 if dim == 2 else 90,
        lambda_volume=1.0,
        lambda_interface=0.2,
    )
    sim.add_cells(cell_type="a", n=n_cells)
    sim.set_adhesion([[0.0, 3.0], [3.0, 1.0]])
    sim.initialize(warn_on_default_init=False)

    start = time.perf_counter()
    sim.run(burn_in_mcs=0, readout_mcs=n_mcs, sampling_interval_mcs=n_mcs)
    elapsed = time.perf_counter() - start
    return n_mcs / elapsed


mcs_per_second(grid=(10, 10, 10), n_mcs=1)  # pay the one-time 3D table cost here

small_2d = mcs_per_second(grid=(30, 30))
large_2d = mcs_per_second(grid=(90, 90))
small_3d = mcs_per_second(grid=(20, 20, 20))

print(f"2D, 30x30:  {small_2d:8.1f} MCS/sec")
print(f"2D, 90x90:  {large_2d:8.1f} MCS/sec  ({small_2d / large_2d:.1f}x slower than 30x30)")
slower_than_2d = small_2d / small_3d
print(f"3D, 20^3:   {small_3d:8.1f} MCS/sec  ({slower_than_2d:.1f}x slower than 2D)")

2D, 30x30:   68579.0 MCS/sec
2D, 90x90:    7934.9 MCS/sec  (8.6x slower than 30x30)
3D, 20^3:     6554.7 MCS/sec  (10.5x slower than 2D)


## Proposal Mode: `uniform` vs `edge_list`

`proposal="uniform"` selects target sites randomly from the interior lattice. Many attempts hit sites surrounded by the same cell type, resulting in no state change. `proposal="edge_list"` restricts selection to border sites between different cells or media.

In packed tissue scenarios where most attempts yield valid moves, the tracking overhead of `edge_list` exceeds the cost of skipped attempts:

| Scenario | uniform | edge_list | Relative Performance |
|---|---|---|---|
| 2D confluent | 464.6 MCS/sec | 374.5 MCS/sec | ~19% slower |
| 3D confluent | 13.83 MCS/sec | 13.59 MCS/sec | ~2% slower |

`edge_list` yields performance gains primarily in sparse environments where a few small cells occupy a large grid dominated by medium.

In [3]:
# Sparse on purpose: a large grid with few, small cells, so most of the
# lattice is medium.
sparse_uniform = mcs_per_second(grid=(150, 150), proposal="uniform")
sparse_edge_list = mcs_per_second(grid=(150, 150), proposal="edge_list")
change = (sparse_edge_list - sparse_uniform) / sparse_uniform * 100

print(f"sparse, uniform:   {sparse_uniform:8.1f} MCS/sec")
print(f"sparse, edge_list: {sparse_edge_list:8.1f} MCS/sec  ({change:+.1f}% vs uniform)")

sparse, uniform:     2405.6 MCS/sec
sparse, edge_list:  80276.3 MCS/sec  (+3237.1% vs uniform)


`edge_list` wins by a wide margin here, the opposite of the confluent result above.
Density is what decides this, not dimension and not grid size on their own. Do not
pick a proposal mode from a rule of thumb. Measure your own configuration with
`mcs_per_second`, the way the two cells above did, before choosing.

## `run_batch`: Parallel Scaling

`sim.run_batch(thetas, ...)` runs independent simulations across available CPU cores. Speedup increases alongside batch size:

| Batch Size | Speedup over Sequential |
|---|---|
| 12 | 5.2x |
| 50 | 9.0x |
| 200 | 10.1x |

Fixed per-call overhead reduces efficiency in small batches. Larger batches distribute work across the thread pool more effectively, yielding higher relative throughput during extensive parameter sweeps or training set generation.

In [4]:
def build_base_sim():
    sim = cpm.CPM(
        grid=(30, 30),
        boundary="periodic",
        seed=1,
        copy_neighborhood="von_neumann",
        connectivity_neighborhood="von_neumann",
    )
    sim.register_cell_type(
        name="a",
        target_volume=30,
        target_interface=40,
        lambda_volume=1.0,
        lambda_interface=0.2,
    )
    sim.add_cells(cell_type="a", n=10)
    sim.set_adhesion([[0.0, 3.0], [3.0, 1.0]])
    sim.initialize(warn_on_default_init=False)
    return sim


def batch_vs_sequential(n, n_mcs=100):
    sim = build_base_sim()
    thetas = [sim.extract_theta() for _ in range(n)]

    start = time.perf_counter()
    for _theta in thetas:
        seq_sim = build_base_sim()
        seq_sim.run(burn_in_mcs=0, readout_mcs=n_mcs, sampling_interval_mcs=n_mcs)
    sequential_elapsed = time.perf_counter() - start

    start = time.perf_counter()
    sim.run_batch(
        thetas, burn_in_mcs=0, readout_mcs=n_mcs, sampling_interval_mcs=n_mcs, master_seed=42
    )
    batch_elapsed = time.perf_counter() - start

    return sequential_elapsed / batch_elapsed


speedup_small = batch_vs_sequential(n=10)
speedup_large = batch_vs_sequential(n=80)
print(f"N=10:  {speedup_small:.1f}x speedup")
print(f"N=80:  {speedup_large:.1f}x speedup")

N=10:  3.6x speedup
N=80:  8.4x speedup


## Readout Cost and Sampling Intervals

Each recorded sample triggers a connected-components flood fill across the lattice to identify lumens and fragmented cells. Benchmark testing at 3D confluent density measures this operation at 5.4 milliseconds per sample compared to 72 milliseconds per MCS.

Sampling on every step (`sampling_interval_mcs=1`) adds a 7.5% overhead per MCS. Setting a larger interval spreads this calculation across multiple cycles, reducing runtime impact while capturing statistically independent states.

## Summary

| Situation | Setting |
|---|---|
| Small 2D exploration | Defaults (`uniform`, no batching) |
| Large 2D or 3D production runs | Start with `uniform`. Measure performance before switching to `edge_list`, as density dictates effectiveness |
| Sparse configurations (few cells on a large grid) | `edge_list` offers potential throughput gains |
| Parameter sweeps or training sets | Use `run_batch` to maximize parallel thread pool efficiency |
| Sample frequency configuration | Increase `sampling_interval_mcs` to reduce duplicate sampling and minimize readout processing time |
| Full state tracking | Enable `include_lattice=True` selectively due to per-sample memory allocation |
| Initial 3D execution | Account for initial setup time during connectivity table construction |

Benchmark specific configurations using `mcs_per_second` to confirm performance gains.